In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('breast-cancer.csv')
df = df.drop(columns = ['id'])

Encoding

In [3]:
df['diagnosis'] = df['diagnosis'].map({'M':1, 'B':0})

Train/Test 

In [4]:
necessary_features = ['diagnosis', 'radius_mean', 'texture_mean', 'concave points_mean', 
                     'radius_worst', 'perimeter_worst', 'area_worst', 
                     'concave points_worst']
all_shuff = df.sample(frac=1.0, random_state=10)
shuff = all_shuff[necessary_features]
train = shuff.iloc[:int(0.8*len(shuff))]
test = shuff.iloc[int(0.8*len(shuff)):]

x_tr = train.drop(columns=['diagnosis']).to_numpy()
y_tr = train['diagnosis'].to_numpy()
x_ts = test.drop(columns=['diagnosis']).to_numpy()
y_ts = test['diagnosis'].to_numpy()

Standardization

In [5]:
mu = x_tr.mean(axis=0)
sigma = x_tr.std(axis=0) + 1e-10
# because I want to save mu and sigma separately at the end
x_train = (x_tr - mu) / sigma
x_test = (x_ts - mu) / sigma


Functions

In [6]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def logistic_regression(X, y, lr, lmbd, epochs):
    w = np.zeros(X.shape[1])
    b = 0.0
    for _ in range(epochs):
        pred = sigmoid(np.dot(X, w) + b)
        # Regularization(L2) is used for w
        # L = cross entropy(w,b) + (lambda / 2m) * w^2
        grad_w = (np.dot(X.T, pred - y) + (lmbd*w)) / len(X)
        grad_b = np.mean(pred - y)
        w = w - lr*grad_w
        b = b - lr*grad_b
    return w, b

def classification(p):
    return np.where(p >= 0.5, 1, 0)

def metrics(y_hat, y):
    tr_pos = np.sum((y_hat==1) & (y==1))
    fl_pos = np.sum((y_hat==1) & (y==0))
    tr_neg = np.sum((y_hat==0) & (y==0))
    fl_neg = np.sum((y_hat==0) & (y==1))

    acc = (tr_pos + tr_neg) / (tr_pos + tr_neg + fl_pos + fl_neg)
    rec = (tr_pos) / (tr_pos + fl_neg)
    prc = (tr_pos) / (tr_pos + fl_pos)
    return acc, rec, prc

Experiment

In [ ]:
import time
rates = [0.1, 0.01, 0.001]
lambdas = [1, 0.1, 0.01, 0]
for r in rates:
    for lmbd in lambdas:
        st = time.time()
        w, b = logistic_regression(x_train, y_tr, r, lmbd, 700)
        ed = time.time()
        pr = sigmoid(np.dot(x_test, w) + b)
        preds = classification(pr)
        met = metrics(preds, y_ts)
        print(f'LR: {r} / Lambda: {lmbd} / Accuracy: {met[0]:.4f} / Recall: {met[1]:.4f} / Precision: {met[2]:.4f} / Time: {ed - st:.5f}')
    print()

LR: 0.1 / Lambda: 1 / Accuracy: 0.9474 / Recall: 0.9333 / Precision: 0.9333 / Time: 0.10004
LR: 0.1 / Lambda: 0.1 / Accuracy: 0.9386 / Recall: 0.9333 / Precision: 0.9130 / Time: 0.06969
LR: 0.1 / Lambda: 0.01 / Accuracy: 0.9386 / Recall: 0.9333 / Precision: 0.9130 / Time: 0.06023
LR: 0.1 / Lambda: 0 / Accuracy: 0.9386 / Recall: 0.9333 / Precision: 0.9130 / Time: 0.06470

LR: 0.01 / Lambda: 1 / Accuracy: 0.9386 / Recall: 0.9333 / Precision: 0.9130 / Time: 0.07000
LR: 0.01 / Lambda: 0.1 / Accuracy: 0.9386 / Recall: 0.9333 / Precision: 0.9130 / Time: 0.07463
LR: 0.01 / Lambda: 0.01 / Accuracy: 0.9386 / Recall: 0.9333 / Precision: 0.9130 / Time: 0.06450
LR: 0.01 / Lambda: 0 / Accuracy: 0.9386 / Recall: 0.9333 / Precision: 0.9130 / Time: 0.07003

LR: 0.001 / Lambda: 1 / Accuracy: 0.9211 / Recall: 0.9333 / Precision: 0.8750 / Time: 0.06902
LR: 0.001 / Lambda: 0.1 / Accuracy: 0.9211 / Recall: 0.9333 / Precision: 0.8750 / Time: 0.07023
LR: 0.001 / Lambda: 0.01 / Accuracy: 0.9211 / Recall: 0.93

Final Train (LR=0.1, Lambda=1)

In [ ]:
final_w, final_b = logistic_regression(x_train, y_tr, 0.1, 1, 700)
pr = sigmoid(np.dot(x_test, final_w) + final_b)
preds = classification(pr)
met = metrics(preds, y_ts)
print(f'Final Results: Accuracy: {met[0]:.4f} / Recall: {met[1]:.4f} / Precision: {met[2]:.4f}')

Saving Model's Parameters

In [ ]:
np.save(r'F:\Network Science\Project\Breast Cancer\the model\model\Logistic Regression\model_w.npy', final_w)
# bias saved as a 1‑element array
np.save(r'F:\Network Science\Project\Breast Cancer\the model\model\Logistic Regression\model_b.npy', np.array([final_b]))
# mean of each feature in the training set
np.save(r'F:\Network Science\Project\Breast Cancer\the model\model\Logistic Regression\model_mu.npy', mu)
# standard deviation of each feature in the training set
np.save(r'F:\Network Science\Project\Breast Cancer\the model\model\Logistic Regression\model_sigma.npy', sigma)

print('model parameters saved in "model" folder!')

K-Nearest Neighbor

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

Standardization

In [ ]:
my_scaler = StandardScaler()

x_train_knn = my_scaler.fit_transform(x_tr)
# fit_transform() learns the mean and standard deviation from the "training" data
x_test_knn = my_scaler.transform(x_ts)
# only transform() using same scaler
y_train_knn = y_tr
y_test_knn = y_ts

Experiment

In [ ]:
k_values = [3, 5, 7, 9, 11]
for k in k_values:
    knn = KNeighborsClassifier(
        n_neighbors= k,
        weights= 'distance',   # controls how neighbors vote; 'default' every neighbor has equal vote;
                                # 'distance' closer neighbors have more weight
        metric= 'minkowski',   # defines how distance is calculated between points. Minkowski is a distance
                                # formula; when p=2 Minkowski becomes Euclidean distance
        p= 2
    )

    # Train & Predict (using sklearn)
    knn.fit(x_train_knn, y_train_knn)
    preds_knn = knn.predict(x_test_knn)

    # Metrics (using sklearn)
    acc_knn = accuracy_score(y_test_knn, preds_knn)
    rec_knn = recall_score(y_test_knn, preds_knn)
    prc_knn = precision_score(y_test_knn, preds_knn)
    
    print(f'K: {k} / Accuracy: {acc_knn:.4f} / Recall: {rec_knn:.4f} / Precision: {prc_knn:.4f}')

Final Train (K = 5)

In [ ]:
knn_final = KNeighborsClassifier(
        n_neighbors= 5,
        weights= 'distance',
        metric= 'minkowski',
        p= 2
    )

knn_final.fit(x_train_knn, y_train_knn)
preds_knn = knn_final.predict(x_test_knn)

acc_knn = accuracy_score(y_test_knn, preds_knn)
rec_knn = recall_score(y_test_knn, preds_knn)
prc_knn = precision_score(y_test_knn, preds_knn)
    
print(f'Accuracy: {acc_knn:.4f} / Recall: {rec_knn:.4f} / Precision: {prc_knn:.4f}')

Saving Model's Parameters

In [ ]:
import joblib

# .pkl = pickle file; Python’s way of serializing objects
joblib.dump(knn_final, r'F:\Network Science\Project\Breast Cancer\the model\model\K Nearest Neighbors\knn_model.pkl')   
joblib.dump(my_scaler, r'F:\Network Science\Project\Breast Cancer\the model\model\K Nearest Neighbors\knn_scaler.pkl')
# saving saclar is the same as saving mu and sigma
print('knn model parameters saved!')

SVM

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, precision_score

Standardization

In [ ]:
my_scaler_svm = StandardScaler()

x_train_svm = my_scaler_svm.fit_transform(x_tr)
x_test_svm = my_scaler_svm.transform(x_ts)
y_train_svm = y_tr
y_test_svm = y_ts

Experiment

In [ ]:
kernels = ['linear', 'rbf', 'poly', 'sigmoid']
c_values = [0.1, 1, 10]
for c in c_values:
    print(f'C: {c}')
    for ker in kernels:
        svm_clf = SVC(
            kernel= ker,
            C= c,   # penalty for misclassification (regularization strength) 
                    # High C → less regularization → model tries to classify training data perfectly
                    # Low C → more regularization → model allows more mistakes but generalizes better
            gamma= 'scale',   # how far the influence of a single training example reaches.
                            # High gamma → each point influences a very small area → can overfit
                            # Low gamma → smoother decision boundary → less overfitting
            probability= True,   # Enables: svm_clf.predict_proba(X) , svm_clf.predict_log_proba(X)
            random_state= 20
        )

        # Train & Predict (using sklearn)
        svm_clf.fit(x_train_svm, y_train_svm)
        preds_svm = svm_clf.predict(x_test_svm)

        # Metrics (using sklearn)
        acc_svm = accuracy_score(y_test_svm, preds_svm)
        rec_svm = recall_score(y_test_svm, preds_svm)
        prc_svm = precision_score(y_test_svm, preds_svm)
        
        print(f'Kernel: {ker} / Accuracy: {acc_svm:.4f} / Recall: {rec_svm:.4f} / Precision: {prc_svm:.4f}')
    print()

Final Train (Kernel = linear, C = 10)

In [ ]:
svm_final = SVC(
    kernel= 'linear',
    C= 10,
    gamma= 'scale',
    probability= True,
    random_state= 20
)

svm_final.fit(x_train_svm, y_train_svm)
preds_svm = svm_final.predict(x_test_svm)

acc_svm = accuracy_score(y_test_svm, preds_svm)
rec_svm = recall_score(y_test_svm, preds_svm)
prc_svm = precision_score(y_test_svm, preds_svm)
        
print(f'Accuracy: {acc_svm:.4f} / Recall: {rec_svm:.4f} / Precision: {prc_svm:.4f}')

Saving Model's Parameters

In [ ]:
import joblib

joblib.dump(svm_final, r'F:\Network Science\Project\Breast Cancer\the model\model\Support Vector Machines\svm_model.pkl')   
joblib.dump(my_scaler_svm, r'F:\Network Science\Project\Breast Cancer\the model\model\Support Vector Machines\svm_scaler.pkl')

print('svm model parameters saved!')